In [1]:
import numpy as npimport yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

SyntaxError: invalid syntax (2831419765.py, line 1)

In [2]:
# --- Download multiple assets to diversify our study ---
# Using a basket of stocks gives us more data and tests generalizability
tickers = ["AAPL", "MSFT", "GOOGL", "JPM", "SPY"]
raw_data = {}

for ticker in tickers:
    raw_data[ticker] = yf.download(ticker, start="2018-01-01", end="2024-01-01",
                                   auto_adjust=True, progress=False)
    print(f"{ticker}: {len(raw_data[ticker])} trading days downloaded")

0

In [ ]:

# Focus on AAPL for our main model, but we'll use SPY as a market benchmark
df = raw_data["AAPL"].copy()
spy = raw_data["SPY"].copy()
python
# --- Base rate analysis (the "naive benchmark") ---
# Your model must beat this number to be worth anything.

df['Daily_return'] = df['Close'].pct_change()
df['Next_return']  = df['Daily_return'].shift(-1)
df['Target']       = (df['Next_return'] > 0).astype(int)

base_rate = df['Target'].dropna().mean()
print(f"Base rate (% of up days): {base_rate:.3%}")
print(f"Naive accuracy if you always predict 'Up': {base_rate:.3%}")
print(f"Your model must exceed:                    {base_rate:.3%}")

In [ ]:

# Also check: is the return distribution normal?
# Most ML models assume some normality. Heavy tails break that assumption.
returns = df['Daily_return'].dropna()
stat, p_value = stats.normaltest(returns)
print(f"\nNormality test p-value: {p_value:.6f}")
print("Returns are NOT normally distributed" if p_value < 0.05 else "Returns appear normal")



In [ ]:
# Kurtosis > 3 means fat tails — more extreme events than a normal distribution predicts
print(f"Kurtosis: {returns.kurtosis():.2f}  (Normal = 0,  Fat tails > 0)")
print(f"Skewness: {returns.skew():.2f}  (Symmetric = 0)")
python
# --- Regime detection: is the market in a trend or sideways? ---
# ML models trained during bull markets often fail in bear markets.
# Understanding regimes helps you know when your model is likely to break.

spy_returns = spy['Close'].pct_change().dropna()



In [ ]:
# Define regimes using 200-day SMA
spy['SMA_200']  = spy['Close'].rolling(200).mean()
spy['Regime']   = np.where(spy['Close'] > spy['SMA_200'], 'Bull', 'Bear')

regime_performance = spy.dropna().groupby('Regime')['Close'].agg(['count']).rename(columns={'count': 'Days'})
print("\nMarket regime breakdown (SPY):")
print(regime_performance)
Phase 2 — Deep Dive: Exploratory Data Analysis
EDA is detective work. You're looking for patterns, anomalies, and relationships that will guide your feature engineering.

